In [42]:
import numpy as np
import random
import time
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd
from sklearn.model_selection import train_test_split

In [43]:
import warnings
warnings.filterwarnings('ignore')

In [44]:
search_space = {
    "logistic_regression": {
        "hyperparams": {
            "C": [0.0001, 0.001, 0.01, 0.1, 1.0, 10, 100],
            "solver": ["liblinear", "lbfgs"]
        },
        "build_fn": lambda hp: LogisticRegression(
            C=hp["C"], 
            solver=hp["solver"],
            max_iter=10000
        )
    },
    "random_forest": {
        "hyperparams": {
            "n_estimators": [5, 10, 20, 30, 50, 100, 200],
            "max_depth": [None, 5, 10, 15, 20, 25, 30],
            "criterion": ["gini", "entropy"]
        },
        "build_fn": lambda hp: RandomForestClassifier(
            n_estimators=hp["n_estimators"],
            max_depth=hp["max_depth"],
            criterion=hp["criterion"],
            random_state=42
        )
    },
    "gradient_boosting": {
        "hyperparams": {
            "n_estimators": [5, 10, 20, 30, 50, 100, 200],
            "learning_rate": [0.00001, 0.0001, 0.001, 0.01, 0.1],
            "max_depth": [3, 5, 7, 9]
        },
        "build_fn": lambda hp: GradientBoostingClassifier(
            n_estimators=hp["n_estimators"],
            learning_rate=hp["learning_rate"],
            max_depth=hp["max_depth"],
            random_state=42
        )
    },
    "svc": {
        "hyperparams": {
            "C": [0.01, 0.1, 1.0, 10],
            "kernel": ["linear", "rbf"],
            "gamma": ["scale", "auto"]
        },
        "build_fn": lambda hp: SVC(
            C=hp["C"],
            kernel=hp["kernel"],
            gamma=hp["gamma"],
            probability=True, # so we can get predict_proba if needed
            random_state=42
        )
    },
    "knn": {
        "hyperparams": {
            "n_neighbors": [1, 3, 5, 7, 9],
            "weights": ["uniform", "distance"],
            "p": [1, 2]  # 1 => Manhattan distance, 2 => Euclidean distance
        },
        "build_fn": lambda hp: KNeighborsClassifier(
            n_neighbors=hp["n_neighbors"],
            weights=hp["weights"],
            p=hp["p"]
        )
    }
}

In [45]:
def random_solution(search_space):
    """
    Generate a random solution from the defined search space.
    A solution is a dict:
    {
      "algo_name": <string>,
      "hyperparams": <dict with each param chosen randomly from the possible range>
    }
    """
    algo_name = random.choice(list(search_space.keys()))
    hyperparams_choices = search_space[algo_name]["hyperparams"]

    chosen_hyperparams = {}
    for param_name, possible_values in hyperparams_choices.items():
        chosen_hyperparams[param_name] = random.choice(possible_values)
    
    return {
        "algo_name": algo_name,
        "hyperparams": chosen_hyperparams
    }


def get_neighbor(solution, search_space):
    """
    Given the current solution, produce a 'neighbor' solution by randomly
    changing either the algorithm or one of the hyperparameters.
    We define the 'neighbor' as:
      - with 50% chance, switch the algorithm to a different one entirely
      - otherwise, pick one hyperparameter and change it to a different 
        permissible value in that hyperparam's range.
    """
    neighbor_sol = {
        "algo_name": solution["algo_name"],
        "hyperparams": solution["hyperparams"].copy()
    }
    
    # Decide whether we switch algorithm or tune a hyperparam
    if random.random() < 0.50:
        # Switch algorithm
        new_algo = random.choice(list(search_space.keys()))
        while new_algo == neighbor_sol["algo_name"]:
            new_algo = random.choice(list(search_space.keys()))
        # pick random hyperparams for the new algorithm
        hyperparams_choices = search_space[new_algo]["hyperparams"]
        chosen_hyperparams = {}
        for param_name, possible_values in hyperparams_choices.items():
            chosen_hyperparams[param_name] = random.choice(possible_values)
        
        neighbor_sol["algo_name"] = new_algo
        neighbor_sol["hyperparams"] = chosen_hyperparams
    else:
        # Switch one hyperparam from the same algorithm
        algo_name = neighbor_sol["algo_name"]
        hyperparams_choices = search_space[algo_name]["hyperparams"]
        param_to_change = random.choice(list(hyperparams_choices.keys()))
        
        current_value = neighbor_sol["hyperparams"][param_to_change]
        possible_values = hyperparams_choices[param_to_change]
        
        # pick a new value for that param (different from the current)
        new_value = random.choice(possible_values)
        while new_value == current_value and len(possible_values) > 1:
            new_value = random.choice(possible_values)
        
        neighbor_sol["hyperparams"][param_to_change] = new_value
    
    return neighbor_sol


def build_model(solution, search_space):
    """
    Given a solution (algorithm + hyperparams),
    build and return the corresponding sklearn model.
    """
    algo_name = solution["algo_name"]
    hp = solution["hyperparams"]
    return search_space[algo_name]["build_fn"](hp)


def evaluate_solution(solution, X, y, cv_folds=5):
    """
    Evaluate a solution by training/testing via cross-validation.
    The evaluation metric is classification accuracy (higher is better).
    
    Returns the negative accuracy for minimization purposes 
    (i.e., we want to *minimize* negative accuracy, i.e. maximize accuracy).
    If you prefer other metrics or a regression setting, adapt accordingly.
    """
    model = build_model(solution, search_space)
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    mean_score = np.mean(scores)
    
    # We transform the objective so that "lower is better" --> negative accuracy
    # Alternatively, we can keep it as 1 - accuracy, or any cost function you prefer.
    cost = -mean_score
    return cost



In [46]:
def simulated_annealing(
    X, y,
    search_space,
    max_iterations=50,
    initial_temperature=1.0,
    min_temperature=0.001,
    alpha=0.85,
    inner_loop=10,
    random_seed=42
):
    """
    Perform Simulated Annealing to find best (algorithm, hyperparameters)
    that maximizes accuracy (or equivalently minimizes negative accuracy).

    Parameters:
    -----------
    X, y:       Training data (features, labels)
    search_space: dict describing algorithms and hyperparams
    max_iterations: how many outer iterations (cooling steps)
    initial_temperature: starting "temperature"
    min_temperature: minimal temperature to stop
    alpha: cooling ratio T <- alpha * T
    inner_loop: how many neighbor solutions to try at each temperature
    random_seed: for reproducibility

    Returns:
    --------
    best_solution, best_cost
    """
    random.seed(random_seed)
    np.random.seed(random_seed)

    # Initialize
    current_solution = random_solution(search_space)
    current_cost = evaluate_solution(current_solution, X, y)
    best_solution = current_solution
    best_cost = current_cost

    T = initial_temperature
    iteration = 0

    # Start SA loop
    while T > min_temperature and iteration < max_iterations:
        print(f"Iteration {iteration + 1}/{max_iterations} - Temperature: {T:.4f} - Current Cost: {current_cost:.4f} - Best Cost: {best_cost:.4f}")
        for inner_iter in range(inner_loop):
            # get neighbor
            neighbor = get_neighbor(current_solution, search_space)
            neighbor_cost = evaluate_solution(neighbor, X, y)

            # if neighbor is better, accept it
            if neighbor_cost < current_cost:
                current_solution = neighbor
                current_cost = neighbor_cost
                print(f"  Inner {inner_iter + 1}/{inner_loop}: Accepted better solution with cost: {current_cost:.4f}")
            else:
                # accept with probability e^(-(neighbor_cost - current_cost)/T)
                cost_diff = neighbor_cost - current_cost
                acceptance_prob = np.exp(-cost_diff / T)
                if random.random() < acceptance_prob:
                    current_solution = neighbor
                    current_cost = neighbor_cost
                    print(f"  Inner {inner_iter + 1}/{inner_loop}: Accepted worse solution with cost: {current_cost:.4f} (prob: {acceptance_prob:.4f})")

            # update global best if needed
            if current_cost < best_cost:
                best_solution = current_solution
                best_cost = current_cost
                print(f"  New Best Found: Cost: {best_cost:.4f}, Algorithm: {best_solution['algo_name']}")

        # cool down
        T = alpha * T
        iteration += 1

    return best_solution, best_cost




# Breast Cancer Dataset

In [47]:
# Load a breast_cancer dataset
X = pd.read_csv('datasets/breast_cancer_preprocessed/X_train.csv', sep=',')
y = pd.read_csv('datasets/breast_cancer_preprocessed/y_train.csv')

X = X.to_numpy()
y = y.to_numpy().ravel()

In [48]:
# Run Simulated Annealing 
start_time = time.time()
best_solution, best_cost = simulated_annealing(
    X, y,
    search_space=search_space,
    max_iterations=100,         # you can tune these parameters
    initial_temperature=2.0,
    min_temperature=0.001,
    alpha=0.85,
    inner_loop=10,
    random_seed=42
)
runtime = time.time() - start_time

# best_cost is negative accuracy (i.e., -accuracy)
best_accuracy = -best_cost

print("Simulated Annealing best solution found:")
print("  Algorithm:", best_solution["algo_name"])
print("  Hyperparameters:", best_solution["hyperparams"])
print(f"  Accuracy: {best_accuracy:.4f}")
print(f"  Total runtime: {runtime:.2f} seconds")

Iteration 1/100 - Temperature: 2.0000 - Current Cost: -0.6785 - Best Cost: -0.6785
  Inner 1/10: Accepted better solution with cost: -0.9497
  New Best Found: Cost: -0.9497, Algorithm: random_forest
  Inner 2/10: Accepted better solution with cost: -0.9599
  New Best Found: Cost: -0.9599, Algorithm: random_forest


/opt/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:794: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 115, in __call__
    score = scorer._score(cached_call, estimator, *args, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 276, in _score
    y_pred = method_caller(estimator, "predict", X)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 73, in _cached_call
    return getattr(estimator, method)(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/neighbors/_classification.py", line 234,

  Inner 4/10: Accepted worse solution with cost: -0.9599 (prob: 1.0000)
  Inner 5/10: Accepted better solution with cost: -0.9797
  New Best Found: Cost: -0.9797, Algorithm: logistic_regression
  Inner 6/10: Accepted worse solution with cost: -0.9797 (prob: 1.0000)
  Inner 7/10: Accepted worse solution with cost: -0.9797 (prob: 1.0000)
  Inner 8/10: Accepted worse solution with cost: -0.6785 (prob: 0.8602)
  Inner 9/10: Accepted worse solution with cost: -0.6785 (prob: 1.0000)
  Inner 10/10: Accepted worse solution with cost: -0.6785 (prob: 1.0000)
Iteration 2/100 - Temperature: 1.7000 - Current Cost: -0.6785 - Best Cost: -0.9797
  Inner 1/10: Accepted worse solution with cost: -0.6785 (prob: 1.0000)
  Inner 2/10: Accepted better solution with cost: -0.9797
  Inner 3/10: Accepted worse solution with cost: -0.9797 (prob: 1.0000)
  Inner 4/10: Accepted worse solution with cost: -0.9497 (prob: 0.9825)
  Inner 5/10: Accepted better solution with cost: -0.9950
  New Best Found: Cost: -0.995

/opt/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:794: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 115, in __call__
    score = scorer._score(cached_call, estimator, *args, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 276, in _score
    y_pred = method_caller(estimator, "predict", X)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 73, in _cached_call
    return getattr(estimator, method)(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/neighbors/_classification.py", line 237,

  Inner 9/10: Accepted worse solution with cost: -0.6785 (prob: 1.0000)
  Inner 10/10: Accepted better solution with cost: -0.9599
Iteration 3/100 - Temperature: 1.4450 - Current Cost: -0.9599 - Best Cost: -0.9950
  Inner 2/10: Accepted worse solution with cost: -0.9397 (prob: 0.9862)
  Inner 3/10: Accepted better solution with cost: -0.9449
  Inner 4/10: Accepted worse solution with cost: -0.7840 (prob: 0.8946)
  Inner 5/10: Accepted better solution with cost: -0.9449
  Inner 6/10: Accepted better solution with cost: -0.9797
  Inner 7/10: Accepted better solution with cost: -0.9849
  Inner 8/10: Accepted worse solution with cost: -0.9599 (prob: 0.9828)
  Inner 9/10: Accepted better solution with cost: -0.9849
  Inner 10/10: Accepted worse solution with cost: -0.9797 (prob: 0.9965)
Iteration 4/100 - Temperature: 1.2282 - Current Cost: -0.9797 - Best Cost: -0.9950
  Inner 1/10: Accepted better solution with cost: -0.9849
  Inner 2/10: Accepted worse solution with cost: -0.9497 (prob: 0.

...

Iteration 44/100 - Temperature: 0.0018 - Current Cost: -0.9950 - Best Cost: -0.9950
Iteration 45/100 - Temperature: 0.0016 - Current Cost: -0.9950 - Best Cost: -0.9950


/opt/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:794: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 115, in __call__
    score = scorer._score(cached_call, estimator, *args, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 276, in _score
    y_pred = method_caller(estimator, "predict", X)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 73, in _cached_call
    return getattr(estimator, method)(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/neighbors/_classification.py", line 237,

Iteration 46/100 - Temperature: 0.0013 - Current Cost: -0.9950 - Best Cost: -0.9950
  Inner 3/10: Accepted worse solution with cost: -0.9899 (prob: 0.0214)
Iteration 47/100 - Temperature: 0.0011 - Current Cost: -0.9899 - Best Cost: -0.9950


/opt/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:794: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 115, in __call__
    score = scorer._score(cached_call, estimator, *args, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 276, in _score
    y_pred = method_caller(estimator, "predict", X)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 73, in _cached_call
    return getattr(estimator, method)(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/neighbors/_classification.py", line 237,

  Inner 2/10: Accepted better solution with cost: -0.9950
Simulated Annealing best solution found:
  Algorithm: logistic_regression
  Hyperparameters: {'C': 1.0, 'solver': 'lbfgs'}
  Accuracy: 0.9950
  Total runtime: 25.08 seconds


# Loan Dataset

In [49]:
# Load a loan dataset
X = pd.read_csv('datasets/loan_preprocessed/X_train.csv', sep=',')
y = pd.read_csv('datasets/loan_preprocessed/y_train.csv')

X = X.to_numpy()
y = y.to_numpy().ravel()

In [50]:
# Get half of the data
X_half, _, y_half, _ = train_test_split(X, y, test_size=0.5, random_state=42)

In [51]:
# Run Simulated Annealing
start_time = time.time()
best_solution, best_cost = simulated_annealing(
    X, y,
    search_space=search_space,
    max_iterations=50,         # you can tune these parameters
    initial_temperature=2.0,
    min_temperature=0.001,
    alpha=0.85,
    inner_loop=10,
    random_seed=42
)
runtime = time.time() - start_time

# best_cost is negative accuracy (i.e., -accuracy)
best_accuracy = -best_cost

print("Simulated Annealing best solution found:")
print("  Algorithm:", best_solution["algo_name"])
print("  Hyperparameters:", best_solution["hyperparams"])
print(f"  Accuracy: {best_accuracy:.4f}")
print(f"  Total runtime: {runtime:.2f} seconds")

Iteration 1/50 - Temperature: 2.0000 - Current Cost: -0.4376 - Best Cost: -0.4376
  Inner 1/10: Accepted better solution with cost: -0.8114
  New Best Found: Cost: -0.8114, Algorithm: random_forest
  Inner 2/10: Accepted worse solution with cost: -0.6396 (prob: 0.9177)


/opt/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:794: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 115, in __call__
    score = scorer._score(cached_call, estimator, *args, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 276, in _score
    y_pred = method_caller(estimator, "predict", X)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 73, in _cached_call
    return getattr(estimator, method)(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/neighbors/_classification.py", line 234,

  Inner 4/10: Accepted better solution with cost: -0.8130
  New Best Found: Cost: -0.8130, Algorithm: random_forest
  Inner 5/10: Accepted worse solution with cost: -0.8047 (prob: 0.9959)
  Inner 6/10: Accepted worse solution with cost: -0.2994 (prob: 0.7767)
  Inner 7/10: Accepted worse solution with cost: -0.2994 (prob: 1.0000)
  Inner 8/10: Accepted worse solution with cost: -0.2994 (prob: 1.0000)


/opt/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:794: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 115, in __call__
    score = scorer._score(cached_call, estimator, *args, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 276, in _score
    y_pred = method_caller(estimator, "predict", X)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 73, in _cached_call
    return getattr(estimator, method)(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/neighbors/_classification.py", line 237,

  Inner 10/10: Accepted better solution with cost: -0.7600
Iteration 2/50 - Temperature: 1.7000 - Current Cost: -0.7600 - Best Cost: -0.8130
  Inner 1/10: Accepted better solution with cost: -0.7647
  Inner 2/10: Accepted worse solution with cost: -0.5686 (prob: 0.8910)
  Inner 3/10: Accepted better solution with cost: -0.6619
  Inner 4/10: Accepted worse solution with cost: -0.5104 (prob: 0.9148)


...

Iteration 45/50 - Temperature: 0.0016 - Current Cost: -0.9890 - Best Cost: -0.9901
  Inner 4/10: Accepted worse solution with cost: -0.9869 (prob: 0.2551)
  Inner 9/10: Accepted better solution with cost: -0.9890
Iteration 46/50 - Temperature: 0.0013 - Current Cost: -0.9890 - Best Cost: -0.9901
  Inner 8/10: Accepted better solution with cost: -0.9891
  Inner 9/10: Accepted better solution with cost: -0.9897
Iteration 47/50 - Temperature: 0.0011 - Current Cost: -0.9897 - Best Cost: -0.9901


/opt/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:794: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 115, in __call__
    score = scorer._score(cached_call, estimator, *args, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 276, in _score
    y_pred = method_caller(estimator, "predict", X)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 73, in _cached_call
    return getattr(estimator, method)(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/neighbors/_classification.py", line 234,

  Inner 5/10: Accepted worse solution with cost: -0.9891 (prob: 0.6039)


/opt/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:794: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 115, in __call__
    score = scorer._score(cached_call, estimator, *args, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 276, in _score
    y_pred = method_caller(estimator, "predict", X)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 73, in _cached_call
    return getattr(estimator, method)(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/sklearn/neighbors/_classification.py", line 234,

Simulated Annealing best solution found:
  Algorithm: gradient_boosting
  Hyperparameters: {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 7}
  Accuracy: 0.9901
  Total runtime: 23762.28 seconds


# Census Income Dataset

In [52]:
# Load a census_income dataset
X = pd.read_csv('datasets/census_income_preprocessed/X_train.csv', sep=',')
y = pd.read_csv('datasets/census_income_preprocessed/y_train.csv')

X = X.to_numpy()
y = y.to_numpy().ravel()

In [53]:
# Get half of the data
X_half, _, y_half, _ = train_test_split(X, y, test_size=0.5, random_state=42)

In [ ]:
# Run Simulated Annealing
start_time = time.time()
best_solution, best_cost = simulated_annealing(
    X, y,
    search_space=search_space,
    max_iterations=50,         # you can tune these parameters
    initial_temperature=2.0,
    min_temperature=0.001,
    alpha=0.85,
    inner_loop=10,
    random_seed=42
)
runtime = time.time() - start_time

# best_cost is negative accuracy (i.e., -accuracy)
best_accuracy = -best_cost

print("Simulated Annealing best solution found:")
print("  Algorithm:", best_solution["algo_name"])
print("  Hyperparameters:", best_solution["hyperparams"])
print(f"  Accuracy: {best_accuracy:.4f}")
print(f"  Total runtime: {runtime:.2f} seconds")

Iteration 1/50 - Temperature: 2.0000 - Current Cost: -0.7247 - Best Cost: -0.7247
  Inner 1/10: Accepted better solution with cost: -0.8396
  New Best Found: Cost: -0.8396, Algorithm: random_forest
  Inner 2/10: Accepted worse solution with cost: -0.8285 (prob: 0.9945)
  Inner 3/10: Accepted worse solution with cost: -0.8095 (prob: 0.9905)
  Inner 4/10: Accepted better solution with cost: -0.8235
  Inner 5/10: Accepted worse solution with cost: -0.8210 (prob: 0.9988)
  Inner 6/10: Accepted better solution with cost: -0.8235
  Inner 7/10: Accepted worse solution with cost: -0.7414 (prob: 0.9598)
  Inner 8/10: Accepted worse solution with cost: -0.7120 (prob: 0.9854)
  Inner 9/10: Accepted better solution with cost: -0.8260
  Inner 10/10: Accepted worse solution with cost: -0.8223 (prob: 0.9982)
Iteration 2/50 - Temperature: 1.7000 - Current Cost: -0.8223 - Best Cost: -0.8396
  Inner 1/10: Accepted worse solution with cost: -0.8080 (prob: 0.9916)
  Inner 2/10: Accepted better solution wi

# Bank Marketing Dataset

In [ ]:
# Load a census_income dataset
X = pd.read_csv('datasets/bank_marketing_preprocessed/X_train.csv', sep=',')
y = pd.read_csv('datasets/bank_marketing_preprocessed/y_train.csv')

X = X.to_numpy()
y = y.to_numpy().ravel()

In [ ]:
# Get half of the data
X_half, _, y_half, _ = train_test_split(X, y, test_size=0.5, random_state=42)

In [ ]:
# Run Simulated Annealing
start_time = time.time()
best_solution, best_cost = simulated_annealing(
    X, y,
    search_space=search_space,
    max_iterations=50,         # you can tune these parameters
    initial_temperature=2.0,
    min_temperature=0.001,
    alpha=0.85,
    inner_loop=10,
    random_seed=42
)
runtime = time.time() - start_time

# best_cost is negative accuracy (i.e., -accuracy)
best_accuracy = -best_cost

print("Simulated Annealing best solution found:")
print("  Algorithm:", best_solution["algo_name"])
print("  Hyperparameters:", best_solution["hyperparams"])
print(f"  Accuracy: {best_accuracy:.4f}")
print(f"  Total runtime: {runtime:.2f} seconds")

# Student Dropout Dataset

In [ ]:
# Load a census_income dataset
X = pd.read_csv('datasets/student_dropout_preprocessed/X_train.csv', sep=',')
y = pd.read_csv('datasets/student_dropout_preprocessed/y_train.csv')

X = X.to_numpy()
y = y.to_numpy().ravel()

In [ ]:
# Run Simulated Annealing
start_time = time.time()
best_solution, best_cost = simulated_annealing(
    X, y,
    search_space=search_space,
    max_iterations=50,         # you can tune these parameters
    initial_temperature=2.0,
    min_temperature=0.001,
    alpha=0.85,
    inner_loop=10,
    random_seed=42
)
runtime = time.time() - start_time

# best_cost is negative accuracy (i.e., -accuracy)
best_accuracy = -best_cost

print("Simulated Annealing best solution found:")
print("  Algorithm:", best_solution["algo_name"])
print("  Hyperparameters:", best_solution["hyperparams"])
print(f"  Accuracy: {best_accuracy:.4f}")
print(f"  Total runtime: {runtime:.2f} seconds")

In [ ]:
automl = autosklearn.classification.AutoSklearnClassifier(time_left_for_this_task=300, seed=42)
automl.fit(X, y)
print("Auto-Sklearn Best Score:", automl.score(X, y))

y_pred_automl = automl.predict(X)

accuracy_automl = automl.score(X, y)

# Extract the best configuration
best_model_automl = automl.show_models()
best_config_automl = automl.get_models_with_weights()

# Print Auto-Sklearn Results
print("Auto-Sklearn Best Solution Found:")
print(f"  Best Configuration: {best_config_automl}")
print(f"  Accuracy: {accuracy_automl:.4f}")